# EV Policy Assistant

## Stage 1: environment checks

Stage 1 setup checks. The complete technical Stage 2 and Stage 3 workflows follows below. Adapted from the course repository at commit `33c2faa22450cde16ead9071f7ce7ecc78ca592a`: Lab 4 cells 6–8, 38, 65, 77, 106 and 150; Exercise 2 cell 4. AI assistance adapted these setup checks; no policy-answering pipeline is implemented here.

Run from the project folder with the project’s Python 3.12 environment. Put your Groq key in the local `.env` file. Notebook outputs should be cleared before committing.

In [ ]:
import os
import sys
import math
from importlib.metadata import version
from dotenv import load_dotenv
import gradio as gr
from langchain.chat_models import init_chat_model
from langchain.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(override=True)

assert sys.version_info[:2] == (3, 12), "Select the project Python 3.12 kernel."
print("Python:", sys.version.split()[0])
for package in ["langchain", "langchain-chroma", "langchain-ollama", "langchain-groq", "gradio", "pypdf", "unstructured"]:
    print(package, version(package))

### Local embeddings

Ollama must be running with `nomic-embed-text` available. This checks one query; it does not build an index.

In [ ]:
embeddings_model = OllamaEmbeddings(model="nomic-embed-text")
query_embedding = embeddings_model.embed_query("EV policy setup check")
assert len(query_embedding) > 0
assert all(math.isfinite(value) for value in query_embedding)
print("Embedding dimensions:", len(query_embedding))

### Groq connection

This sends a small test prompt to Groq and uses the account’s API allowance. A missing key stops the check; it is not a successful connection test.

In [ ]:
if not os.environ.get("GROQ_API_KEY"):
    raise ValueError("Add GROQ_API_KEY to the local .env file and rerun the setup cells.")

model_name = "openai/gpt-oss-120b"
llm = init_chat_model(model_name, model_provider="groq", temperature=0, max_tokens=256, timeout=30, max_retries=0)
response = llm.invoke("Reply with only OK.")
assert response.content, "Groq returned an empty response."
print(response.content)

## Shared page loader

Stages 2 and 3 reuse this loader and the course `Document` pattern. Run this cell before either ingestion stage. Plain extraction remains the default; Tamil Nadu's manifest selects layout extraction on three table pages. The layout option only collapses whitespace padding and records that choice in metadata; it does not rewrite amounts or OCR text.

In [ ]:
import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_policy_pages(manifest, review, state):
    sources = [source for source in manifest['sources'] if source['state'] == state]
    if not sources or len({source['source_id'] for source in sources}) != len(sources):
        raise ValueError('Missing or duplicate source records.')

    ocr_pages = {}
    for page in review['pages']:
        key = (page['source_id'], page['pdf_page'])
        if key in ocr_pages:
            raise ValueError(f'Duplicate OCR page: {key}')
        ocr_pages[key] = page

    documents = []
    for source in sources:
        source_id = source['source_id']
        if file_hash(source['filename']) != source['sha256']:
            raise ValueError(f'PDF hash mismatch: {source_id}')
        reader = PdfReader(source['filename'])
        if len(reader.pages) != source['pdf_page_count']:
            raise ValueError(f'PDF page count mismatch: {source_id}')
        page_numbers = source['candidate_pdf_pages']
        if not page_numbers or len(set(page_numbers)) != len(page_numbers):
            raise ValueError(f'Missing or duplicate candidate pages: {source_id}')
        source_ocr = {page for sid, page in ocr_pages if sid == source_id}
        uses_ocr = 'ocr_review_file' in source
        if uses_ocr and source_ocr != set(page_numbers):
            raise ValueError(f'OCR coverage mismatch: {source_id}')
        if source_ocr and not uses_ocr:
            raise ValueError(f'Unexpected OCR records: {source_id}')

        for pdf_page in page_numbers:
            if not isinstance(pdf_page, int) or not 1 <= pdf_page <= len(reader.pages):
                raise ValueError(f'Invalid physical page: {source_id}, {pdf_page}')
            metadata = {key: source[key] for key in (
                'state', 'policy_year', 'source_id', 'document_title', 'document_date',
                'official_url', 'team_verified', 'accepted_for_ingestion',
                'current_benefit_availability', 'current_entitlement_answers_allowed'
            )}
            metadata.update({
                'source': source['filename'],
                'source_sha256': source['sha256'],
                'pdf_page': pdf_page,
                'page_id': f'{source_id}:p{pdf_page}',
                'page_role': source.get('page_roles', {}).get(str(pdf_page), 'policy_text'),
                'verification_cutoff': manifest['verification_cutoff'],
                'text_status': 'pdf_extraction',
            })
            for key in ('supplements_source_id', 'clarifies_source_id', 'amends_source_id', 'amended_section',
                        'effective_from', 'effective_to', 'benefit_scope', 'document_date_precision'):
                if key in source:
                    metadata[key] = source[key]

            if uses_ocr:
                page = ocr_pages[(source_id, pdf_page)]
                if page['source_pdf'] != source['filename'] or page['source_sha256'] != source['sha256']:
                    raise ValueError(f'OCR source mismatch: {source_id}, {pdf_page}')
                if page.get('page_role', metadata['page_role']) != metadata['page_role']:
                    raise ValueError(f'OCR page role mismatch: {source_id}, {pdf_page}')
                if file_hash(page['proposed_text']) != page['proposed_sha256']:
                    raise ValueError(f'OCR proposal hash mismatch: {source_id}, {pdf_page}')
                page_text = Path(page['proposed_text']).read_text(encoding='utf-8')
                metadata.update({
                    'text_file': page['proposed_text'],
                    'text_status': 'ai_proposal',
                    'team_verified': False,
                    'accepted_for_ingestion': False,
                })
            else:
                pdf = reader.pages[pdf_page - 1]
                if pdf_page in source.get('layout_pdf_pages', []):
                    raw_text = pdf.extract_text(extraction_mode='layout')
                    # Collapse layout padding, keeping row order and line breaks.
                    page_text = '\n'.join(' '.join(line.split()) for line in raw_text.splitlines())
                    page_text = re.sub(r'\n{3,}', '\n\n', page_text).strip()
                    metadata['text_status'] = 'pdf_layout_whitespace_normalized'
                else:
                    page_text = pdf.extract_text()
            if not page_text or not page_text.strip():
                raise ValueError(f'Empty page text: {source_id}, {pdf_page}')
            metadata['text_sha256'] = hashlib.sha256(page_text.encode('utf-8')).hexdigest()
            documents.append(Document(page_content=page_text, metadata=metadata))
    return documents


## Stage 2: Maharashtra ingestion and chunk checks

The user authorized AI implementation of technical Stage 2. This replaces the earlier worked examples with one complete ingestion path. Manual source review and the team's two evaluation examples remain deferred; no current-benefit availability is inferred.

Course reuse at commit `33c2faa22450cde16ead9071f7ce7ecc78ca592a`: Lab 4 cells 65–66 for load/inspect order; Exercise 2 cell 9 for `Document` and metadata dictionaries; Lab 4 cell 77 for recursive splitting with start indices. `pypdf.PdfReader`, source hashes, OCR sidecars and evidence links are additions for page citations.

Run the shared page loader, then all Stage 2 cells in order from the project folder. They run without Stage 1, API keys, Groq or Ollama. Output files are candidate data for development, not a verified answer corpus.


In [ ]:
source_manifest = json.loads(Path('data/source_manifest.json').read_text())
ocr_review = json.loads(Path('data/ocr/maharashtra/review.json').read_text())

page_documents = load_policy_pages(source_manifest, ocr_review, 'Maharashtra')
print('Loaded pages:', len(page_documents))
print(dict(Counter(doc.metadata['source_id'] for doc in page_documents)))

### Evidence relationships

Each record retains one physical page. The links below connect a table to its conditions or a sentence to its continuation; they do not merge citations. Related page IDs are JSON strings so metadata stays compatible with scalar-only vector-store fields later.

The August old wording and distribution list stay in the page audit but are excluded from draft answer chunks. Base page 19 keeps its unaffected provisions and a section-specific link to the toll replacement. All records remain unverified even when their text is suitable for chunk tests.


In [ ]:
policy_id = 'maharashtra_policy_2025-05-23'
june_id = 'maharashtra_operational_guidelines_2025-06-19'
july_id = 'maharashtra_operational_guidelines_2025-07-28'
corrigendum_id = 'maharashtra_corrigendum_2025-08-29'
page_by_id = {doc.metadata['page_id']: doc for doc in page_documents}

page_pairs = [(policy_id, 17, 18), (policy_id, 18, 19), (policy_id, 19, 20),
              (policy_id, 21, 22), (policy_id, 22, 23), (policy_id, 23, 24),
              (policy_id, 24, 25), (june_id, 2, 3)]
related_pages = {page_id: [] for page_id in page_by_id}
for source_id, first, second in page_pairs:
    first_id, second_id = f'{source_id}:p{first}', f'{source_id}:p{second}'
    related_pages[first_id].append(second_id)
    related_pages[second_id].append(first_id)

replacement_page_id = f'{corrigendum_id}:p2'
related_pages[f'{policy_id}:p19'].append(replacement_page_id)
related_pages[replacement_page_id].append(f'{policy_id}:p19')
page_by_id[f'{policy_id}:p18'].metadata['conditions_page_id'] = f'{policy_id}:p19'
page_by_id[f'{policy_id}:p19'].metadata.update({
    'amended_section': '4.2(1)',
    'replacement_page_id': replacement_page_id,
})
page_by_id[f'{corrigendum_id}:p1'].metadata['replacement_page_id'] = replacement_page_id
page_by_id[replacement_page_id].metadata['replaces_page_id'] = f'{corrigendum_id}:p1'

for doc in page_documents:
    doc.metadata['related_page_ids'] = json.dumps(related_pages[doc.metadata['page_id']])
    doc.metadata['include_in_draft_chunks'] = doc.metadata['page_role'] not in (
        'old_wording_and_amendment_scope', 'distribution_list_only'
    )

draft_pages = [doc for doc in page_documents if doc.metadata['include_in_draft_chunks']]
print('Pages for draft chunks:', len(draft_pages))
print('Audit-only pages:', [doc.metadata['page_id'] for doc in page_documents
                            if not doc.metadata['include_in_draft_chunks']])


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
policy_chunks = text_splitter.split_documents(draft_pages)
for chunk in policy_chunks:
    content_hash = hashlib.sha256(chunk.page_content.encode('utf-8')).hexdigest()[:12]
    chunk.metadata['chunk_id'] = f"{chunk.metadata['page_id']}:c{chunk.metadata['start_index']}:{content_hash}"

print('Draft chunks:', len(policy_chunks))
print(dict(Counter(chunk.metadata['source_id'] for chunk in policy_chunks)))


### Technical checks

These checks concern faithful extraction and chunk boundaries. They are not the team's evaluation questions, proof of policy validity or a substitute for human review. The two incentive tables fit inside individual default chunks; if source text changes and breaks that property, this cell fails before export.


In [ ]:
expected_pages = {policy_id: set(range(16, 26)), june_id: {1, 2, 3, 4},
                  july_id: {1}, corrigendum_id: {1, 2, 3}}
assert len(page_documents) == len(page_by_id) == 18
for source_id, pages in expected_pages.items():
    assert {doc.metadata['pdf_page'] for doc in page_documents
            if doc.metadata['source_id'] == source_id} == pages
assert len(draft_pages) == 16
assert len({chunk.metadata['chunk_id'] for chunk in policy_chunks}) == len(policy_chunks)

source_ids = set(expected_pages)
chunks_by_page = {doc.metadata['page_id']: [] for doc in draft_pages}
for doc in page_documents:
    assert doc.page_content.strip()
    assert doc.metadata['source'].endswith('.pdf')
    assert doc.metadata['team_verified'] is False
    assert doc.metadata['accepted_for_ingestion'] is False
    assert doc.metadata['current_benefit_availability'] == 'not_verified'
    assert doc.metadata['current_entitlement_answers_allowed'] is False
    assert doc.metadata['verification_cutoff'] == '2026-09-23'
    assert all(page_id in page_by_id for page_id in json.loads(doc.metadata['related_page_ids']))
    for field in ('supplements_source_id', 'clarifies_source_id', 'amends_source_id'):
        if field in doc.metadata:
            assert doc.metadata[field] in source_ids
    if doc.metadata['text_status'] == 'ai_proposal':
        assert doc.metadata['source'] != doc.metadata['text_file']
        assert file_hash(doc.metadata['text_file']) == doc.metadata['text_sha256']

for chunk in policy_chunks:
    page = page_by_id[chunk.metadata['page_id']]
    start = chunk.metadata['start_index']
    assert start >= 0 and page.page_content[start:start + len(chunk.page_content)] == chunk.page_content
    assert all(chunk.metadata[key] == value for key, value in page.metadata.items())
    assert all(isinstance(value, (str, int, float, bool)) for value in chunk.metadata.values())
    chunks_by_page[chunk.metadata['page_id']].append(chunk)

# No non-whitespace page text may disappear during splitting.
for page_id, chunks in chunks_by_page.items():
    page_text = page_by_id[page_id].page_content
    covered = set()
    for chunk in chunks:
        start = chunk.metadata['start_index']
        covered.update(range(start, start + len(chunk.page_content)))
    assert all(index in covered or character.isspace() for index, character in enumerate(page_text))


def compact(text):
    return ''.join(text.split())


def containing_chunk(page_id, text):
    return next(chunk for chunk in chunks_by_page[page_id]
                if compact(text) in compact(chunk.page_content))


table_2_page = page_by_id[f'{policy_id}:p18'].page_content
table_2 = table_2_page[table_2_page.index('Table 2:'):].strip()
table_2_chunk = containing_chunk(f'{policy_id}:p18', table_2)
expected_rows = [
    '1 e-2W (L1 & L2) 10% 1,00,000 10,000',
    '2 e-3W (L5M) 10% 15,000 30,000',
    '3 e-3W goods carrier (L5N) 15% 10,000 30,000',
    '4 e-4W cars (M1) (Non-Transport vehicle) 10% 10,000 1,50,000',
    '5 e-4W cars (M1) (Transport vehicle) 15% 25,000 2,00,000',
    '6 e-4W Light Goods Carrier (N1) 15% 10,000 1,00,000',
    '7 e-buses (M3, M4) (STU) 10% 1,500 20,00,000',
    '8 e-buses (M3, M4) (non-STU) 10% 1,500 20,00,000',
    '9 e-4W goods carrier (N2, N3) 15% 1,000 20,00,000',
    '10 e-Agricultural Tractors and combined harvesters (A) 15% 1,000 1,50,000',
]
assert all(compact(row) in compact(table_2_chunk.page_content) for row in expected_rows)
conditions_page_id = table_2_chunk.metadata['conditions_page_id']
conditions_text = page_by_id[conditions_page_id].page_content
conditions_note = conditions_text[conditions_text.index('Note- Demand incentives'):conditions_text.index('2)')]
conditions_chunk = containing_chunk(conditions_page_id, conditions_note)

table_3_page = page_by_id[f'{policy_id}:p20'].page_content
table_3 = table_3_page[table_3_page.index('Table 3:'):table_3_page.index('3)')]
table_3_chunk = containing_chunk(f'{policy_id}:p20', table_3)
for phrase in ('minimum 4 charging points installed', 'DC 50 kW to 250 kW',
               'Up to 15% INR 5.00 Lakhs 1,000', 'minimum 2 charging points installed',
               '(250 to > 500 kW)', 'Up to 15% INR 10.00 Lakhs 500',
               'does not include land and any ancillary cost'):
    assert compact(phrase) in compact(table_3_chunk.page_content)

june_start_id, june_end_id = f'{june_id}:p2', f'{june_id}:p3'
june_start = page_by_id[june_start_id].page_content
june_start = june_start[june_start.index('६.'):]
june_end = page_by_id[june_end_id].page_content.split('७.')[0]
june_start_chunk = containing_chunk(june_start_id, june_start)
june_end_chunk = containing_chunk(june_end_id, june_end)
assert june_end_id in json.loads(june_start_chunk.metadata['related_page_ids'])
assert june_start_id in json.loads(june_end_chunk.metadata['related_page_ids'])

replacement_text = page_by_id[replacement_page_id].page_content
replacement_text = replacement_text[replacement_text.index('याऐवजी'):replacement_text.index('२. सदर')]
replacement_chunk = containing_chunk(replacement_page_id, replacement_text)
assert 'संबंधित विभाग /' in replacement_chunk.page_content and 'प्राधिकरणास' in replacement_chunk.page_content
assert replacement_chunk.metadata['page_role'] == 'replacement_wording_and_authority'
assert replacement_chunk.metadata['amends_source_id'] == policy_id
assert page_by_id[f'{policy_id}:p19'].metadata['replacement_page_id'] == replacement_page_id
assert page_by_id[f'{july_id}:p1'].metadata['clarifies_source_id'] == june_id
assert all(chunk.metadata['page_role'] not in ('old_wording_and_amendment_scope', 'distribution_list_only')
           for chunk in policy_chunks)

check_names = ['18_unique_pages', 'original_pdf_citations', 'source_and_proposal_hashes',
               'relationships_resolve', '16_draft_pages', 'metadata_survives_splitting',
               'no_nonwhitespace_text_lost', 'table_2_all_10_rows_and_conditions',
               'table_3_rows_units_and_exclusion', 'june_clause_6_continuation',
               'august_replacement_and_audit_exclusions', 'unverified_status_preserved']
print('Technical checks passed:', len(check_names))


In [ ]:
output_dir = Path('data/processed/maharashtra')
output_dir.mkdir(parents=True, exist_ok=True)
for filename, documents in [('pages.jsonl', page_documents), ('chunks.jsonl', policy_chunks)]:
    records = [{'page_content': doc.page_content, 'metadata': doc.metadata} for doc in documents]
    (output_dir / filename).write_text(
        ''.join(json.dumps(record, ensure_ascii=False) + '\n' for record in records), encoding='utf-8'
    )

stage2_report = {
    'checked_at_utc': datetime.now(timezone.utc).isoformat(),
    'technical_status': 'passed',
    'team_acceptance': 'deferred_not_verified',
    'current_benefit_availability': source_manifest['current_benefit_availability'],
    'verification_cutoff': source_manifest['verification_cutoff'],
    'page_count': len(page_documents),
    'draft_page_count': len(draft_pages),
    'chunk_count': len(policy_chunks),
    'chunks_by_source': dict(Counter(chunk.metadata['source_id'] for chunk in policy_chunks)),
    'splitter': {'chunk_size': 1000, 'chunk_overlap': 200, 'add_start_index': True},
    'checks_passed': check_names,
    'evidence_chunks': {
        'table_2': table_2_chunk.metadata['chunk_id'],
        'table_2_conditions': conditions_chunk.metadata['chunk_id'],
        'table_3': table_3_chunk.metadata['chunk_id'],
        'june_clause_6_start': june_start_chunk.metadata['chunk_id'],
        'june_clause_6_end': june_end_chunk.metadata['chunk_id'],
        'august_replacement': replacement_chunk.metadata['chunk_id'],
    },
    'artifact_sha256': {name: file_hash(output_dir / name) for name in ('pages.jsonl', 'chunks.jsonl')},
    'notebook_sha256': file_hash('EV Policy Assistant.ipynb'),
}
(output_dir / 'stage2_checks.json').write_text(json.dumps(stage2_report, ensure_ascii=False, indent=2) + '\n')
print('Saved:', output_dir)
print('Technical Stage 2 passed; team review and current-benefit verification remain pending.')


### Stage boundary

Technical Stage 2 ends here: candidate pages, draft chunks and observed checks are saved under `data/processed/maharashtra/`. Re-running these cells replaces those derived files; it does not edit the PDFs, OCR proposals or source-review flags. The output is not a Chroma index and no embeddings or policy answers are generated.

This loader deliberately uses AI-proposed OCR. Team acceptance must later select and hash genuinely checked text before treating it as verified; changing a reviewer flag alone cannot promote a proposal. Evidence links must also be followed when retrieval is implemented in Stage 6. Keeping links in metadata is not itself an answer-grounding mechanism.

The team's two independently authored/verified examples and current-benefit evidence remain outstanding. Stage 3 below covers the selected second jurisdiction, Tamil Nadu. See `PROGRESS.md` for the checkpoint and AI-assistance record.


## Stage 3: Tamil Nadu ingestion

The user selected Tamil Nadu and requested completion of technical Stage 3. Run the shared page loader, then these cells in order; Stage 1/2 execution and model services are not needed. Manual source review and the two team-written evaluation cases remain deferred.

Reuse: Stage 2's `load_policy_pages`, Exercise 2 cell 9's `Document` metadata and Lab 4 cell 77's recursive splitter. The official 2023 booklet has 28 physical pages; pages 6–27 contain the policy. The separate December 2025 notification supplies the later motor-vehicle-tax period. Covers, blanks and contents are excluded. All citations retain original physical page numbers.

The default text extraction separates Tamil Nadu table columns. Layout extraction on pages 17, 19 and 20 keeps rows together after removing whitespace padding. This is a source-specific extraction setting, not a new OCR pipeline. See `data/policies/tamil_nadu/SOURCE_REVIEW.md` for observed limitations.

In [ ]:
source_manifest = json.loads(Path('data/source_manifest.json').read_text())
tn_pages = load_policy_pages(source_manifest, {'pages': []}, 'Tamil Nadu')
tn_policy_id = 'tamil_nadu_policy_2023'
tn_tax_id = 'tamil_nadu_motor_vehicle_tax_2025-12-29'
tn_page_by_id = {doc.metadata['page_id']: doc for doc in tn_pages}
tn_related = {page_id: [] for page_id in tn_page_by_id}

# Adjacent pages retain cross-page context; each citation still names one page.
for page in range(6, 27):
    first, second = f'{tn_policy_id}:p{page}', f'{tn_policy_id}:p{page + 1}'
    tn_related[first].append(second)
    tn_related[second].append(first)
tax_page_id = f'{tn_tax_id}:p1'
tn_related[f'{tn_policy_id}:p16'].append(tax_page_id)
tn_related[tax_page_id].append(f'{tn_policy_id}:p16')
for doc in tn_pages:
    metadata = doc.metadata
    metadata['related_page_ids'] = json.dumps(tn_related[metadata['page_id']])
    metadata['include_in_draft_chunks'] = True
    if metadata['source_id'] == tn_policy_id:
        metadata['policy_period_page_id'] = f'{tn_policy_id}:p27'
    if metadata['page_id'] in (f'{tn_policy_id}:p16', f'{tn_policy_id}:p17'):
        metadata['demand_incentive_end_as_printed'] = '2025-12-31'
        metadata['demand_incentive_extension_status'] = 'not_established'
    if metadata['page_id'] == f'{tn_policy_id}:p16':
        metadata['road_tax_update_page_id'] = tax_page_id
        metadata['updated_section'] = '4.2.3.1(A) motor vehicle tax only'
    if metadata['page_id'] == f'{tn_policy_id}:p17':
        metadata['conditions_page_id'] = metadata['page_id']
        metadata['incentive_period_page_id'] = f'{tn_policy_id}:p16'
    if metadata['page_id'] in (f'{tn_policy_id}:p19', f'{tn_policy_id}:p20'):
        metadata['conditions_page_id'] = f'{tn_policy_id}:p19'

print('Tamil Nadu pages:', len(tn_pages))
print(dict(Counter(doc.metadata['source_id'] for doc in tn_pages)))

In [ ]:
# Keep public/private charging headings with their own rows.
tn_separators = ['\n5.2.1', '\n5.2.2', '\n\n', '\n', ' ', '']
tn_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True, separators=tn_separators
)
tn_chunks = tn_splitter.split_documents(tn_pages)
for chunk in tn_chunks:
    content_hash = hashlib.sha256(chunk.page_content.encode('utf-8')).hexdigest()[:12]
    chunk.metadata['chunk_id'] = f"{chunk.metadata['page_id']}:c{chunk.metadata['start_index']}:{content_hash}"
print('Tamil Nadu draft chunks:', len(tn_chunks))

### Stage 3 technical checks

Check extraction and chunk boundaries, not model answers. The demand table must retain all five category/amount/limit rows. Conditions and expiry stay linked to the table; the later tax order must not become an extension of every benefit. Both jurisdictions must keep distinct IDs and state metadata.

In [ ]:
assert len(tn_pages) == len(tn_page_by_id) == 23
assert {doc.metadata['pdf_page'] for doc in tn_pages if doc.metadata['source_id'] == tn_policy_id} == set(range(6, 28))
assert {doc.metadata['pdf_page'] for doc in tn_pages if doc.metadata['source_id'] == tn_tax_id} == {1}
assert len({chunk.metadata['chunk_id'] for chunk in tn_chunks}) == len(tn_chunks)
tn_chunks_by_page = {page_id: [] for page_id in tn_page_by_id}
for doc in tn_pages:
    assert doc.page_content.strip()
    assert doc.metadata['source'].endswith('.pdf')
    assert doc.metadata['official_url'].startswith('https://')
    assert file_hash(doc.metadata['source']) == doc.metadata['source_sha256']
    assert doc.metadata['state'] == 'Tamil Nadu' and doc.metadata['policy_year'] == 2023
    assert doc.metadata['team_verified'] is False and doc.metadata['accepted_for_ingestion'] is False
    assert doc.metadata['current_benefit_availability'] == 'not_verified'
    assert doc.metadata['current_entitlement_answers_allowed'] is False
    assert doc.metadata['verification_cutoff'] == '2026-09-23'
    assert all(page_id in tn_page_by_id for page_id in json.loads(doc.metadata['related_page_ids']))
    for key in ('conditions_page_id', 'policy_period_page_id', 'incentive_period_page_id', 'road_tax_update_page_id'):
        if key in doc.metadata:
            assert doc.metadata[key] in tn_page_by_id
for chunk in tn_chunks:
    page = tn_page_by_id[chunk.metadata['page_id']]
    start = chunk.metadata['start_index']
    assert start >= 0 and page.page_content[start:start + len(chunk.page_content)] == chunk.page_content
    assert all(chunk.metadata[key] == value for key, value in page.metadata.items())
    assert all(isinstance(value, (str, int, float, bool)) for value in chunk.metadata.values())
    tn_chunks_by_page[chunk.metadata['page_id']].append(chunk)
for page_id, chunks in tn_chunks_by_page.items():
    covered = set()
    for chunk in chunks:
        covered.update(range(chunk.metadata['start_index'], chunk.metadata['start_index'] + len(chunk.page_content)))
    assert all(i in covered or char.isspace() for i, char in enumerate(tn_page_by_id[page_id].page_content))


def tn_find_chunk(page_id, text):
    return next(chunk for chunk in tn_chunks_by_page[page_id]
                if ''.join(text.split()) in ''.join(chunk.page_content.split()))


tn_demand_text = tn_page_by_id[f'{tn_policy_id}:p17'].page_content
tn_demand_table = tn_demand_text.split('This shall be subject')[0]
tn_demand_chunk = tn_find_chunk(f'{tn_policy_id}:p17', tn_demand_table)
for row in ('Private e-Cycles* - 20% of cost up to 5,000 6,000',
            'Commercial e-2Wheelers 10,000/ kWh 30,000 6,000',
            'Commercial e-3Wheelers (autos/ Light Goods Carriers) 10,000/ kWh 40,000 15,000',
            'Commercial e-4Wheelers (Cabs/Goods Vehicles) 10,000/ kWh 1,50,000 3,000',
            'Commercial e-Buses 20,000/ kWh 10,00,000 300'):
    assert ''.join(row.split()) in ''.join(tn_demand_chunk.page_content.split())
assert tn_demand_chunk.metadata['conditions_page_id'] == f'{tn_policy_id}:p17'
assert tn_demand_chunk.metadata['incentive_period_page_id'] == f'{tn_policy_id}:p16'
for condition in ('manufactured, sold and registered in the State complying with FAME II',
                  'Only e-cycles procured for initiatives under Government programmes',
                  'Direct Benefit Transfer (DBT)', 'less than 0.25 kW', 'less than 25 km/h'):
    tn_find_chunk(f'{tn_policy_id}:p17', condition)
tn_find_chunk(f'{tn_policy_id}:p16', 'following incentives till 31.12.2025')

# Public and private stations have separate caps/counts despite the same fast-charger amount.
tn_charging_text = tn_page_by_id[f'{tn_policy_id}:p19'].page_content
tn_public = tn_charging_text[tn_charging_text.index('5.2.1'):tn_charging_text.index('5.2.2')]
tn_public_chunk = tn_find_chunk(f'{tn_policy_id}:p19', tn_public)
for row in ('25% subsidy', 'Fast Charging Station Up to Rs 10,00,000 200',
            'Slow Charging Station Up to Rs 1,00,000 500'):
    assert ''.join(row.split()) in ''.join(tn_public_chunk.page_content.split())
tn_private = tn_charging_text[tn_charging_text.index('The first 50 private'):]
tn_private_chunk = tn_find_chunk(f'{tn_policy_id}:p19', tn_private)
assert '25%' in tn_private_chunk.page_content
assert 'FastChargingStationUptoRs10,00,00050' in ''.join(tn_private_chunk.page_content.split())
tn_find_chunk(f'{tn_policy_id}:p19', 'cost of land (purchase/lease cost)')
tn_find_chunk(f'{tn_policy_id}:p19', 'at least 75%')
tn_swap_chunk = tn_find_chunk(f'{tn_policy_id}:p20', tn_page_by_id[f'{tn_policy_id}:p20'].page_content)
for phrase in ('first 200', '25%', 'Rs. 2 lakh per station'):
    assert phrase in tn_swap_chunk.page_content

tn_tax_text = tn_page_by_id[tax_page_id].page_content
tn_tax_clause = tn_tax_text[tn_tax_text.index('In exercise'):tn_tax_text.index('DHEERAJ')]
tn_tax_chunk = tn_find_chunk(tax_page_id, tn_tax_clause)
for phrase in ('both Transport and Non-Transport', '1st January 2026', '31st December 2027'):
    assert phrase in tn_tax_chunk.page_content
assert tn_tax_chunk.metadata['benefit_scope'] == 'motor_vehicle_tax_only'
assert tn_tax_chunk.metadata['effective_from'] == '2026-01-01'
assert tn_tax_chunk.metadata['effective_to'] == '2027-12-31'
assert tn_tax_chunk.metadata['supplements_source_id'] == tn_policy_id
assert tn_page_by_id[f'{tn_policy_id}:p16'].metadata['road_tax_update_page_id'] == tax_page_id
assert tn_demand_chunk.metadata['demand_incentive_extension_status'] == 'not_established'
tn_find_chunk(f'{tn_policy_id}:p27', '5 years from the date of the policy notification')

mh_records = [json.loads(line) for line in Path('data/processed/maharashtra/chunks.jsonl').read_text().splitlines()]
assert len(mh_records) == 53 and {record['metadata']['state'] for record in mh_records} == {'Maharashtra'}
assert {record['metadata']['chunk_id'] for record in mh_records}.isdisjoint(chunk.metadata['chunk_id'] for chunk in tn_chunks)
tn_check_names = ['23_unique_pages', 'source_hashes_and_original_citations', 'physical_page_numbers',
                  'metadata_and_status_preserved', 'relationship_targets_resolve', 'no_nonwhitespace_text_lost',
                  'five_demand_rows_and_conditions', 'demand_expiry_not_extended_by_tax_order',
                  'public_private_charging_tables', 'battery_swapping_cap_and_count',
                  'tax_order_scope_dates_and_link', 'distinct_jurisdiction_ids']
print('Stage 3 technical checks passed:', len(tn_check_names))

In [ ]:
tn_output_dir = Path('data/processed/tamil_nadu')
tn_output_dir.mkdir(parents=True, exist_ok=True)
for filename, documents in [('pages.jsonl', tn_pages), ('chunks.jsonl', tn_chunks)]:
    records = [{'page_content': doc.page_content, 'metadata': doc.metadata} for doc in documents]
    (tn_output_dir / filename).write_text(
        ''.join(json.dumps(record, ensure_ascii=False) + '\n' for record in records), encoding='utf-8'
    )
tn_report = {
    'checked_at_utc': datetime.now(timezone.utc).isoformat(), 'technical_status': 'passed',
    'team_acceptance': 'deferred_not_verified', 'current_benefit_availability': 'not_verified',
    'verification_cutoff': source_manifest['verification_cutoff'],
    'page_count': len(tn_pages), 'chunk_count': len(tn_chunks),
    'chunks_by_source': dict(Counter(chunk.metadata['source_id'] for chunk in tn_chunks)),
    'splitter': {'chunk_size': 1000, 'chunk_overlap': 200, 'add_start_index': True, 'separators': tn_separators},
    'checks_passed': tn_check_names,
    'evidence_chunks': {'demand_table': tn_demand_chunk.metadata['chunk_id'],
                        'public_charging': tn_public_chunk.metadata['chunk_id'],
                        'private_charging': tn_private_chunk.metadata['chunk_id'],
                        'battery_swapping': tn_swap_chunk.metadata['chunk_id'],
                        'motor_vehicle_tax': tn_tax_chunk.metadata['chunk_id']},
    'artifact_sha256': {name: file_hash(tn_output_dir / name) for name in ('pages.jsonl', 'chunks.jsonl')},
    'notebook_sha256': file_hash('EV Policy Assistant.ipynb'),
    'source_manifest_sha256': file_hash('data/source_manifest.json'),
}
(tn_output_dir / 'stage3_checks.json').write_text(json.dumps(tn_report, ensure_ascii=False, indent=2) + '\n')
print('Saved:', tn_output_dir)
print('Technical Stage 3 passed; manual acceptance and policy-status gaps remain pending.')

### Stage 3 boundary

The two pilot jurisdictions now have separate, reproducible page/chunk artifacts with consistent citations and state fields. This tests ingestion isolation; actual filtered retrieval belongs to Stage 6. No embeddings, index or answers are created here.

The official tax notification gives a specific 2026–2027 period. That does not prove the booklet's purchase subsidies, registration/permit waivers or every other benefit are currently available. Team review, amendment completeness and two independently authored Tamil Nadu evaluation cases remain pending. Stage 4 will select and prepare the central scheme; it has not started.